In [ ]:
# train_contrastive.py
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import sys, os

sys.path.append("..")  
from helpers.image_encoder import ImageEncoder
from helpers.text_encoder import TextEncoder
from helpers.loss import contrastive_loss 
from helpers.fusion import FusionModule
from helpers.projection import ProjectionModule
# from helpers.dataset import MultimodalDataset

from transformers import AutoTokenizer, AutoModel

from model.model import ICDRetrievalModel
from data.dataset import create_dataloaders



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

# ---------------------------
# Initialize modules
# ---------------------------

image_encoder_model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')  # pretrained DINO backbone
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
text_encoder_model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")   # pretrained MedCLIP text encoder

image_encoder = ImageEncoder(image_encoder_model)
text_encoder = TextEncoder(text_encoder_model)

# Define fusion model
img_dim = 384          # DINO CLS
text_dim = 762         # MedCLIP
projected_dim = 512        # projected fused embedding -> can also try with 256 and see if it helps with contrastive learning
concatenated_dim = img_dim + text_dim
fusion_module = FusionModule(img_dim, text_dim, method="concat").to(device)

# Projection for fused image+text embeddings and ICD embeddings to common space
query_projection = ProjectionModule(input_dim=concatenated_dim, projected_dim=projected_dim, method="linear").to(device)
icd_projection = ProjectionModule(input_dim=text_dim, projected_dim=projected_dim, method="linear").to(device)


# Initialize model 
model = ICDRetrievalModel(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    fusion_module=fusion_module,
    projection_module=query_projection,
    icd_projection_module=icd_projection,
    embedding_dim=projected_dim,
    fusion_method="concat"
).to(device)


In [ ]:

def train_one_epoch(
    dataloader, 
    model: ICDRetrievalModel, 
    optimizer, 
    device,
    temperature=0.07
):
    """
    Train for one epoch with multi-label contrastive loss.
    
    Handles:
    - Multi-label ICDs per image
    - Missing image or text modalities
    
    Args:
        dataloader: DataLoader yielding (images, image_texts, icd_texts_list)
                    - images: list of image paths or None
                    - image_texts: list of query text strings or None
                    - icd_texts_list: list of lists of ICD description strings
        model: ICDRetrievalModel instance
        optimizer: PyTorch optimizer
        device: torch device
        temperature: Temperature for contrastive loss scaling
        
    Returns:
        Average loss over the epoch
    """
    # Set projection modules to train mode
    model.query_projection.proj.train()
    model.icd_projection.proj.train()
    
    total_loss = 0
    
    for batch in dataloader:
        # Unpack batch
        images, image_texts, icd_texts_list = batch
        batch_size = len(images)
        
        # -----------------------
        # Encode and fuse query embeddings
        # -----------------------
        query_embeddings = []
        
        for i in range(batch_size):
            # Get query embedding (handles None for missing modalities)
            image_path = images[i] if images[i] is not None else None
            query_text = image_texts[i] if image_texts[i] is not None else None
            
            # Forward pass through model
            outputs = model(image_path=image_path, query_text=query_text)
            query_emb = outputs['query_embedding']  # [1, embedding_dim], already normalized
            query_embeddings.append(query_emb)
        
        # Stack all query embeddings
        x_fused_proj = torch.cat(query_embeddings, dim=0).to(device)  # [B, embedding_dim]
        
        # -----------------------
        # Encode ICD embeddings
        # -----------------------
        icd_embeddings = []
        icd_image_map = []  # Maps ICD index -> image index for target construction
        
        for img_idx, icd_texts in enumerate(icd_texts_list):
            for icd_text in icd_texts:
                # Encode ICD description
                icd_text_emb = model.encode_icd_text(icd_text).to(device)
                icd_emb = model.icd_projection.project(icd_text_emb)
                icd_emb = F.normalize(icd_emb, dim=-1)
                
                icd_embeddings.append(icd_emb)
                icd_image_map.append(img_idx)
        
        # Stack all ICD embeddings
        x_icd_proj = torch.cat(icd_embeddings, dim=0).to(device)  # [N_icd_total, embedding_dim]
        
        # -----------------------
        # Compute similarity matrix
        # -----------------------
        # Cosine similarity: [B, N_icd_total] (embeddings already normalized)
        sim_matrix = torch.matmul(x_fused_proj, x_icd_proj.T)
        sim_matrix = sim_matrix / temperature
        
        # -----------------------
        # Multi-label contrastive loss
        # -----------------------
        # Create target mask: [B, N_icd_total] -> 1 if ICD belongs to image
        target_mask = torch.zeros_like(sim_matrix, device=device)
        for icd_idx, img_idx in enumerate(icd_image_map):
            target_mask[img_idx, icd_idx] = 1.0
        
        # Compute loss: average negative log-likelihood over positives
        log_prob = F.log_softmax(sim_matrix, dim=1)
        loss = -(target_mask * log_prob).sum(dim=1) / target_mask.sum(dim=1).clamp(min=1.0)
        loss = loss.mean()
        
        # -----------------------
        # Backpropagation
        # -----------------------
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


In [ ]:

# Only optimize the projection layers (encoders are frozen)
optimizer = torch.optim.AdamW([
    {'params': model.query_projection.proj.parameters()},
    {'params': model.icd_projection.proj.parameters()}
], lr=1e-4)

# Load ICD descriptions (you'll need to provide this)
icd_code_to_desc = load_icd_descriptions()  # Your function

# Create dataloaders
train_dl, val_dl, test_dl = create_dataloaders(
    json_path="path/to/data.jsonl",
    mimic_root="/datasets/MIMIC-CXR",
    icd_code_to_desc=icd_code_to_desc,
    batch_size=16
)

num_epochs = 10
# Train
for epoch in range(num_epochs):
    loss = train_one_epoch(train_dl, model, optimizer, device)
    print(f"Epoch {epoch}: Loss = {loss:.4f}")
# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    avg_loss = train_one_epoch(
        dataloader=train_dl,
        model=model,
        optimizer=optimizer,
        device=device,
        temperature=0.07
    )
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")